# Visual context detector

Cuts a video into scenes, then asks **Qwen2.5-VL-7B** which haptic categories are visible in each scene: `gunshot`, `explosion`, `car_crash`, `smash`, `vehicle`, `engine_start`.

A scene can carry more than one label, so a car hitting a wall is both `car_crash` and `smash`, and a tank being started is both `engine_start` and `vehicle`. The first four are bang scenes: in the haptic pipeline a short one may confirm a sharp sound attack. `vehicle` and `engine_start` only label events. Short cuts stay separate. A cut shorter than one second is widened only for the frames the model sees, because Qwen needs at least a few frames; the saved start and end stay the real cut. Short scenes are sampled more densely so a flash is more likely to land in a frame. About 16 frames are sent per scene, which fits an L4.

These spans are a **hint**, not haptic event times. The saved start and end are the scene cut, not a bang time. The haptic notebook should keep using the audio detector for start, peak, and end. An audio bang with no picture span is still kept.

**Runtime:** Colab L4 GPU, high RAM. Run the cells top to bottom. The model is loaded once.

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
!pip install -q scenedetect qwen-vl-utils accelerate

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get("HF_TOKEN"))

In [ ]:
from pathlib import Path
from google.colab import files

print("Upload a video (mp4, mov, mkv). About 5 minutes is a comfortable length.")
uploaded = files.upload()
video_name = next(iter(uploaded))
video_path = Path("/content") / video_name
print("Uploaded:", video_path)

In [ ]:
import json
from scenedetect import ContentDetector, SceneManager, open_video

def scene_chunks(path, threshold=30.0):
    """Cut the video at scene changes. Short cuts stay separate so a brief shot is not buried in a long label."""
    video = open_video(str(path))
    manager = SceneManager()
    manager.add_detector(ContentDetector(threshold=threshold))
    manager.detect_scenes(video)
    scenes = manager.get_scene_list()
    if scenes:
        chunks = [{"start": s[0].seconds, "end": s[1].seconds} for s in scenes]
    else:
        chunks = [{"start": 0.0, "end": video.duration.seconds}]
    chunks = [c for c in chunks if c["end"] > c["start"]]
    duration = float(getattr(video.duration, "seconds", 0.0) or 0.0)
    return chunks, duration

scene_chunks_list, video_duration = scene_chunks(video_path)
shortest = min((c["end"] - c["start"] for c in scene_chunks_list), default=0.0)
print(f"{len(scene_chunks_list)} scenes (short cuts are kept). Video is {video_duration:.1f}s. Shortest cut is {shortest:.3f}s.")
print(json.dumps(scene_chunks_list[:8], indent=2))

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

print("Loading", MODEL_ID)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("Model ready.")

In [ ]:
import json
import re
import torch
from qwen_vl_utils import process_vision_info

# Same names as haptic_gt/context/visual_scenes.py.
VISUAL_CATEGORIES = ["gunshot", "explosion", "car_crash", "smash", "vehicle", "engine_start"]
MAX_FRAMES = 16
MIN_LOOK_SEC = 1.0

PROMPT = (
    "Classify this video segment for haptic context.\n"
    "List every category that is visibly present. A segment may have more than one.\n"
    "- gunshot: a person firing a gun, visible gunfire\n"
    "- explosion: a blast, fireball, artillery impact\n"
    "- car_crash: a vehicle colliding with another vehicle, a wall, or an object\n"
    "- smash: something breaking or being smashed, glass shattering, an object hit hard\n"
    "- vehicle: a car, truck, or tank driving or idling\n"
    "- engine_start: a vehicle engine being started, ignition, first rev\n"
    "car_crash may be listed together with smash; engine_start together with vehicle.\n"
    "Reply with JSON only, no other text. Example for a car hitting a wall: "
    '{"categories": ["car_crash", "smash"]}\n'
    "Use an empty list if none of these are visible: "
    '{"categories": []}'
)

def parse_categories(text):
    """Accept only category names from the model's JSON. Ignore the example placeholder."""
    names = []
    for match in re.finditer(r"\{.*?\}|\[.*?\]", text, flags=re.DOTALL):
        try:
            payload = json.loads(match.group(0))
        except json.JSONDecodeError:
            continue
        if isinstance(payload, list):
            names.extend(payload)
        elif isinstance(payload, dict):
            raw = payload.get("categories", payload.get("category", []))
            if isinstance(raw, str):
                raw = [raw]
            if isinstance(raw, list):
                names.extend(raw)
    if not names:
        names = [part.strip() for part in text.strip().lower().split(",")]
    found = []
    for name in names:
        name = str(name).strip().lower().replace(" ", "_").replace("-", "_")
        if name in VISUAL_CATEGORIES and name not in found:
            found.append(name)
    return found

def sample_fps(start, end):
    """Denser frames on a short cut, never more than MAX_FRAMES for the 7B model."""
    duration = max(end - start, MIN_LOOK_SEC)
    if duration <= 4.0:
        target = 8.0
    elif duration <= 8.0:
        target = 4.0
    else:
        target = 2.0
    return round(min(target, MAX_FRAMES / duration), 2)

def look_window(start, end, duration):
    """Widen a tiny cut so the model receives at least one second of frames."""
    if end - start >= MIN_LOOK_SEC:
        return start, end
    look_start = max(0.0, start - (MIN_LOOK_SEC - (end - start)) / 2.0)
    look_end = look_start + MIN_LOOK_SEC
    if duration and look_end > duration:
        look_end = duration
        look_start = max(0.0, look_end - MIN_LOOK_SEC)
    return look_start, look_end

def classify_scene(path, start, end):
    look_start, look_end = look_window(start, end, video_duration)
    fps = sample_fps(look_start, look_end)
    messages = [{
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": str(path),
                "video_start": look_start,
                "video_end": look_end,
                "fps": fps,
                "min_frames": 2,
                "max_frames": MAX_FRAMES,
                "cap_pixels_per_frame": True,
            },
            {"type": "text", "text": PROMPT},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    if not video_inputs:
        raise RuntimeError(f"no frames decoded for {look_start:.3f}-{look_end:.3f}s")
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=80)
    reply = processor.batch_decode(
        generated[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )[0]
    return reply, parse_categories(reply), fps, look_start, look_end

visual_context = []
failed = 0
print(f"Classifying {len(scene_chunks_list)} scenes.")
for i, scene in enumerate(scene_chunks_list, start=1):
    start, end = scene["start"], scene["end"]
    try:
        reply, categories, fps, look_start, look_end = classify_scene(video_path, start, end)
    except Exception as exc:
        failed += 1
        print(f"\n[{i}/{len(scene_chunks_list)}] {start:.1f}s-{end:.1f}s FAILED: {exc}")
        continue
    label = ", ".join(categories) if categories else "none"
    looked = "" if (look_start, look_end) == (start, end) else f" (looked {look_start:.1f}-{look_end:.1f})"
    print(f"\n[{i}/{len(scene_chunks_list)}] {start:.1f}s-{end:.1f}s @ {fps:g} fps -> {label}{looked}")
    print(" ", reply.replace("\n", " ")[:180])
    for category in categories:
        visual_context.append({
            "category": category,
            "start_sec": round(start, 3),
            "end_sec": round(end, 3),
            "source": "qwen_vl",
        })

print(f"\n{len(visual_context)} visual spans. {failed} scenes failed and were skipped.")
if not visual_context:
    print("No spans were kept. Read the lines above: FAILED means that cut could not be decoded; none means the reply had no category name.")
print(json.dumps(visual_context, indent=2))

In [ ]:
import json
from pathlib import Path
from google.colab import files

out_path = Path("/content/visual_context.json")
payload = {
    "video": video_path.name,
    "note": "Visual hints only. One scene may list more than one category, with the same start and end. Audio detection still owns start, peak, and end. Do not drop an audio bang that has no span here.",
    "spans": visual_context,
}
out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote", out_path, "with", len(visual_context), "spans")
if not visual_context:
    print("spans is empty. Re-run the classify cell and read its printout before downloading.")
files.download(str(out_path))